# Chapter 3: SQL & Data Acquisition
## CompTIA SecAI+ Study Lab

**Objectives:**
- Build and query a relational database using SQLite
- Write SQL queries: SELECT, WHERE, GROUP BY, HAVING, JOINs
- Simulate an ETL (Extract, Transform, Load) pipeline
- Use subqueries for advanced data retrieval

> **Exam Tip:** CompTIA SecAI+ heavily tests SQL concepts. Know the order of SQL clause execution: FROM → WHERE → GROUP BY → HAVING → SELECT → ORDER BY. HAVING filters *after* aggregation; WHERE filters *before* it.


---
## Exercise 1: Create a SQLite Database with Two Tables

A **relational database** organizes data into tables linked by keys:
- **Primary Key (PK):** Uniquely identifies each row in a table
- **Foreign Key (FK):** References a primary key in another table to establish a relationship

> **Exam Tip:** Know the three types of relationships: One-to-One, One-to-Many, Many-to-Many. The customers→orders relationship here is **One-to-Many** (one customer can have many orders).


In [ ]:
import sqlite3
import pandas as pd

# Create an in-memory SQLite database
conn = sqlite3.connect(':memory:')
cursor = conn.cursor()

# TODO: Create the 'customers' table with columns:
#   customer_id (INTEGER, PRIMARY KEY), name (TEXT), email (TEXT), region (TEXT)
cursor.execute("""-- TODO: Write CREATE TABLE customers statement here
""")

# TODO: Create the 'orders' table with columns:
#   order_id (INTEGER, PRIMARY KEY), customer_id (INTEGER, FOREIGN KEY → customers),
#   product (TEXT), amount (REAL), order_date (TEXT)
cursor.execute("""-- TODO: Write CREATE TABLE orders statement here
""")

# TODO: Insert at least 4 customers and 8 orders (mix of customers)

conn.commit()
print("Tables created and data inserted.")

# Verify with a simple SELECT
print(pd.read_sql("SELECT * FROM customers", conn))
print(pd.read_sql("SELECT * FROM orders", conn))


In [ ]:
# SOLUTION
# import sqlite3
# import pandas as pd
#
# conn = sqlite3.connect(':memory:')
# cursor = conn.cursor()
#
# cursor.execute("""
#     CREATE TABLE customers (
#         customer_id INTEGER PRIMARY KEY,
#         name TEXT NOT NULL,
#         email TEXT,
#         region TEXT
#     )
# """)
#
# cursor.execute("""
#     CREATE TABLE orders (
#         order_id INTEGER PRIMARY KEY,
#         customer_id INTEGER,
#         product TEXT,
#         amount REAL,
#         order_date TEXT,
#         FOREIGN KEY (customer_id) REFERENCES customers(customer_id)
#     )
# """)
#
# customers = [
#     (1, 'Alice', 'alice@example.com', 'North'),
#     (2, 'Bob',   'bob@example.com',   'South'),
#     (3, 'Carol', 'carol@example.com', 'East'),
#     (4, 'Dave',  'dave@example.com',  'West'),
# ]
# cursor.executemany("INSERT INTO customers VALUES (?,?,?,?)", customers)
#
# orders = [
#     (101, 1, 'Laptop',  1200.00, '2024-01-15'),
#     (102, 1, 'Mouse',     25.00, '2024-01-20'),
#     (103, 2, 'Keyboard',  75.00, '2024-02-01'),
#     (104, 2, 'Monitor',  350.00, '2024-02-10'),
#     (105, 3, 'Laptop',  1100.00, '2024-02-15'),
#     (106, 3, 'Webcam',    55.00, '2024-03-01'),
#     (107, 4, 'Mouse',     25.00, '2024-03-05'),
#     (108, 1, 'Headset',  150.00, '2024-03-10'),
# ]
# cursor.executemany("INSERT INTO orders VALUES (?,?,?,?,?)", orders)
# conn.commit()
# print(pd.read_sql("SELECT * FROM customers", conn))
# print(pd.read_sql("SELECT * FROM orders", conn))


---
## Exercise 2: SELECT, WHERE, GROUP BY, HAVING

Core SQL clauses every data analyst must master:
- `WHERE` — filters individual rows *before* grouping
- `GROUP BY` — aggregates rows sharing common values
- `HAVING` — filters *aggregated* groups (like WHERE but post-aggregation)

> **Exam Tip:** `HAVING` is only used with `GROUP BY`. A common trap question swaps WHERE and HAVING — remember: WHERE cannot use aggregate functions like COUNT() or SUM().


In [ ]:
# Assumes the connection 'conn' from Exercise 1 is still active
# If starting fresh, re-run Exercise 1 solution first

# TODO: Query 1 — SELECT all orders where amount > 100, ordered by amount DESC
q1 = """-- TODO"""
print("Orders over $100:")
print(pd.read_sql(q1, conn))

# TODO: Query 2 — Total amount spent per customer_id using GROUP BY + SUM
q2 = """-- TODO"""
print("\nTotal per customer:")
print(pd.read_sql(q2, conn))

# TODO: Query 3 — Customers who have placed more than 1 order (use HAVING COUNT > 1)
q3 = """-- TODO"""
print("\nCustomers with more than 1 order:")
print(pd.read_sql(q3, conn))


In [ ]:
# SOLUTION
# q1 = "SELECT * FROM orders WHERE amount > 100 ORDER BY amount DESC"
# print("Orders over $100:")
# print(pd.read_sql(q1, conn))
#
# q2 = """
#     SELECT customer_id, SUM(amount) AS total_spent, COUNT(*) AS num_orders
#     FROM orders
#     GROUP BY customer_id
# """
# print("\nTotal per customer:")
# print(pd.read_sql(q2, conn))
#
# q3 = """
#     SELECT customer_id, COUNT(*) AS num_orders
#     FROM orders
#     GROUP BY customer_id
#     HAVING COUNT(*) > 1
# """
# print("\nCustomers with more than 1 order:")
# print(pd.read_sql(q3, conn))


---
## Exercise 3: INNER JOIN and LEFT JOIN

JOINs combine rows from two or more tables based on a related column:
- **INNER JOIN** — returns only rows where the key exists in *both* tables
- **LEFT JOIN** — returns *all* rows from the left table, with NULLs for non-matching right table rows

> **Exam Tip:** Know all four join types: INNER, LEFT (OUTER), RIGHT (OUTER), FULL OUTER. A LEFT JOIN keeping only NULLs from the right side is how you find records with *no match* (e.g., customers with no orders).


In [ ]:
# TODO: INNER JOIN — Get customer name + order details for every order
inner_join_sql = """-- TODO: JOIN customers and orders on customer_id"""
print("INNER JOIN result:")
print(pd.read_sql(inner_join_sql, conn))

# TODO: LEFT JOIN — Show ALL customers, including those with no orders
#       (If you have a customer with no orders, they'll appear with NULL order fields)
left_join_sql = """-- TODO: LEFT JOIN customers → orders"""
print("\nLEFT JOIN result (all customers):")
print(pd.read_sql(left_join_sql, conn))


In [ ]:
# SOLUTION
# inner_join_sql = """
#     SELECT c.name, c.region, o.product, o.amount, o.order_date
#     FROM orders o
#     INNER JOIN customers c ON o.customer_id = c.customer_id
#     ORDER BY c.name
# """
# print("INNER JOIN result:")
# print(pd.read_sql(inner_join_sql, conn))
#
# left_join_sql = """
#     SELECT c.name, c.region, o.order_id, o.product, o.amount
#     FROM customers c
#     LEFT JOIN orders o ON c.customer_id = o.customer_id
#     ORDER BY c.name
# """
# print("\nLEFT JOIN result (all customers):")
# print(pd.read_sql(left_join_sql, conn))


---
## Exercise 4: Simulate an ETL Pipeline

**ETL** = Extract, Transform, Load — the foundation of data engineering:
1. **Extract** — Read raw data from a source (CSV, API, database)
2. **Transform** — Clean, standardize, rename, filter, enrich
3. **Load** — Write transformed data to the target (data warehouse, DB)

> **Exam Tip:** Know ETL vs ELT. In ELT, raw data is loaded *first*, then transformed inside the destination (common in cloud data warehouses like Snowflake or BigQuery). ETL transforms *before* loading.


In [ ]:
import pandas as pd
import io

# ---- EXTRACT ----
# Raw CSV source data (messy)
raw_csv = """cust_id,CUSTOMER NAME,  email ,purchase_amt,date
C001,  alice smith ,alice@email.com,1200.50,2024-01-15
C002,BOB JONES,bob@email.com, 75.00 ,2024-01-22
C003,  Carol White  ,carol@email.com,350.00,2024-02-01
C004,dave brown,,25.00,2024-02-10
C005,Eve Davis,eve@email.com,900.00,2024-03-01
"""

# TODO: EXTRACT — Read raw_csv into df_raw using pd.read_csv(io.StringIO(...))
df_raw = None
print("--- EXTRACTED ---")
# print(df_raw)

# TODO: TRANSFORM — Apply all of these steps:
#   1. Strip whitespace from all column names (use str.strip())
#   2. Rename columns: 'cust_id' → 'customer_id', 'CUSTOMER NAME' → 'name', 'purchase_amt' → 'amount'
#   3. Strip whitespace from string columns 'name' and 'email'
#   4. Title-case the 'name' column
#   5. Convert 'amount' to float (strip spaces first)
#   6. Convert 'date' to datetime
#   7. Fill missing emails with 'unknown@placeholder.com'
df_clean = None  # TODO: apply transforms to df_raw.copy()
print("\n--- TRANSFORMED ---")
# print(df_clean)

# TODO: LOAD — Write df_clean to SQLite table 'purchases' using df.to_sql()
print("\n--- LOADED ---")
# Verify
# print(pd.read_sql("SELECT * FROM purchases", conn))


In [ ]:
# SOLUTION
# import pandas as pd
# import io
#
# raw_csv = """cust_id,CUSTOMER NAME,  email ,purchase_amt,date
# C001,  alice smith ,alice@email.com,1200.50,2024-01-15
# C002,BOB JONES,bob@email.com, 75.00 ,2024-01-22
# C003,  Carol White  ,carol@email.com,350.00,2024-02-01
# C004,dave brown,,25.00,2024-02-10
# C005,Eve Davis,eve@email.com,900.00,2024-03-01
# """
#
# # EXTRACT
# df_raw = pd.read_csv(io.StringIO(raw_csv))
# print("--- EXTRACTED ---")
# print(df_raw)
#
# # TRANSFORM
# df_clean = df_raw.copy()
# df_clean.columns = df_clean.columns.str.strip()
# df_clean.rename(columns={'cust_id': 'customer_id', 'CUSTOMER NAME': 'name', 'purchase_amt': 'amount'}, inplace=True)
# df_clean['name'] = df_clean['name'].str.strip().str.title()
# df_clean['email'] = df_clean['email'].str.strip()
# df_clean['amount'] = df_clean['amount'].astype(str).str.strip().astype(float)
# df_clean['date'] = pd.to_datetime(df_clean['date'])
# df_clean['email'].fillna('unknown@placeholder.com', inplace=True)
# print("\n--- TRANSFORMED ---")
# print(df_clean)
#
# # LOAD
# df_clean.to_sql('purchases', conn, if_exists='replace', index=False)
# print("\n--- LOADED ---")
# print(pd.read_sql("SELECT * FROM purchases", conn))


---
## Exercise 5 (Challenge): Subquery

A **subquery** is a query nested inside another query. It can appear in the SELECT, FROM, or WHERE clause. Subqueries are essential for multi-step logic without creating temp tables.

> **Exam Tip:** Know the difference between a *correlated subquery* (references the outer query, runs once per row) and a *non-correlated subquery* (runs independently first, then feeds the outer query). Correlated subqueries are slower but more flexible.


In [ ]:
# Using the orders and customers tables from Exercise 1
# (Re-run Exercise 1 solution if needed)

# TODO: Write a query that returns the names and total spend of customers
#       whose total spending is ABOVE the average order amount.
#       Use a subquery to calculate the average.
#
# Structure hint:
#   SELECT c.name, SUM(o.amount) AS total_spent
#   FROM orders o JOIN customers c ...
#   GROUP BY ...
#   HAVING SUM(o.amount) > (SELECT AVG(amount) FROM orders)

subquery_sql = """-- TODO"""

print("Customers spending above average:")
print(pd.read_sql(subquery_sql, conn))


In [ ]:
# SOLUTION
# subquery_sql = """
#     SELECT c.name, SUM(o.amount) AS total_spent
#     FROM orders o
#     JOIN customers c ON o.customer_id = c.customer_id
#     GROUP BY c.customer_id, c.name
#     HAVING SUM(o.amount) > (SELECT AVG(amount) FROM orders)
#     ORDER BY total_spent DESC
# """
# print("Customers spending above average:")
# print(pd.read_sql(subquery_sql, conn))
#
# # Bonus: What IS the average?
# avg = pd.read_sql("SELECT AVG(amount) AS avg_order FROM orders", conn)
# print(f"\nAverage order amount: ${avg['avg_order'][0]:.2f}")
